# AI Reasoning Sandbox

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://elearningindustry.com/wp-content/uploads/2022/05/shutterstock_1162149235.jpg"> 
</p>
</div>

## Description :

An advanced AI-powered application designed to generate multiple distinct and logically sound reasoning paths for any given prompt. 

It leverages OpenAI's powerful language models to produce well-structured, professional, and diverse perspectives, helping users explore complex problems through different reasoning approaches. 

The output is formatted in clean Markdown, making it easy to read and share.

### Key features include:  

- Support for a variety of reasoning styles to cater to different analytical needs.  

- Generation of multiple reasoning paths per prompt to ensure comprehensive analysis.  

- Clear structure with titles, stepwise logic, and concise conclusions for each reasoning path.  

- Easy integration with frontend environments, enabling direct display of AI-generated content.

### 🧠 Reasoning Styles Supported

| **Style**           | **Description**                                             | **Example**                                                                 |
|---------------------|-------------------------------------------------------------|------------------------------------------------------------------------------|
| **🕵️‍♂️ Abductive**        | Infers the most likely explanation from incomplete data     | The grass is wet. It probably rained last night.                            |
| **🔗 Analogical**       | Draws conclusions by comparing similar situations           | Just as a car needs fuel, our brains need rest to perform well.             |
| **🧩 Analytical**       | Breaks down complex ideas into simpler components           | To solve the bug, I isolated each module to find the error.                 |
| **⚙️ Causal**           | Identifies cause-and-effect relationships                   | Skipping sleep caused lower focus during the meeting.                       |
| **❓ Counterfactual**   | Explores "what if" scenarios and hypothetical outcomes      | If I had studied more, I would have passed the exam.                        |
| **🎨 Creative**         | Uses analogies, metaphors, or imaginative ideas             | Managing a team is like conducting an orchestra—each member plays a part.   |
| **🧐 Critical**         | Evaluates ideas with logic, evidence, and skepticism        | The report sounds convincing, but the data source is unreliable.            |
| **🤔 Critical Thinking**| Applies objective analysis and rational judgment            | Instead of reacting emotionally, I examined the pros and cons first.        |
| **🧪 Deductive**        | From general premises to a specific conclusion              | All humans are mortal. Socrates is a human. Therefore, Socrates is mortal.  |
| **⚖️ Ethical**          | Considers moral values and consequences                     | Lying may help now, but honesty builds long-term trust.                     |
| **🔍 Inductive**        | From specific observations to general patterns              | The sun rose every day this week. It will likely rise tomorrow too.         |
| **💭 Introspective**    | Uses self-reflection and internal insight for reasoning     | I feel unmotivated lately—maybe I need a short break to reset.              |
| **🛠 Practical**        | Focuses on what works effectively in real situations        | Let’s choose the faster method—it meets our deadline.                       |
| **🧳 Pragmatic**        | Focuses on practical consequences and real-world usefulness | This tool saves time, so it's worth the investment.                         |
| **🪞 Reflective**       | Involves thinking deeply about past experiences             | I noticed my stress rose after skipping workouts—I'll avoid that again.     |
| **📋 Systematic**       | Follows a structured, step-by-step method                   | I used a checklist to troubleshoot the network issue one layer at a time.   |


## Step 1: Environment Setup and Installation

This cell handles initial setup for the notebook:

- Installs dependencies from `requirements/ai_reasoning_sandbox.requirements.txt`.

- Retries installation up to 3 times on failure.

- Loads environment variables from `.env` using `python-dotenv`.

- Ensures `OPENAI_API_KEY` is set before continuing.

After setup, it clears the output and confirms success.


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
from dotenv import load_dotenv
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]
PROJECT_NAME = "ai_reasoning_sandbox"
REQUIREMENTS_FILE = f"{PROJECT_NAME}.requirements.txt"


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system(f"pip install -r requirements/{REQUIREMENTS_FILE}")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)


install_requirements()
clear_output()
setup_env()
print("🚀 Setup complete. Continue to the next cell.")

## Step 2: Data Models for Reasoning and Query Records

- Defines structured data models using Pydantic.

- `ReasoningPaths`: stores a reasoning path with a title, step-by-step points, and a conclusion.

- `QueryRecord`: represents a user query with an ID, prompt, optional reasoning style, and multiple reasoning paths.

- Ensures clear data structure and validation for responses.


In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class ReasoningPaths(BaseModel):
    title: str
    steps: List[str]
    conclusion: str

class QueryRecord(BaseModel):
    id: str
    prompt : str
    reasoning_style: Optional[str]
    paths: List[ReasoningPaths]

## Step 3: AI Reasoning Sandbox: Function Overview

- The `__init__` method initializes the class by setting up the OpenAI client, model name, temperature, and an empty history list.  

- The `system_prompt` method generates a base system message and optionally tailors it to a specific reasoning style.  

- The `user_prompt` method constructs detailed user instructions to ask the AI for multiple distinct reasoning paths formatted in markdown.  

- The `generate_response` method sends the system and user prompts to the OpenAI API and returns the generated markdown response with reasoning paths. 

- The `add_query` method calls `generate_response`, saves the query details including a unique ID, prompt, and reasoning style to history, and returns the raw response text.  

- The `get_history` method returns the full list of stored query records as dictionaries.  

- The `clear_history` method clears all saved queries and confirms that the history has been emptied.  


In [ ]:
import os
import json
import re
from typing import List, Optional, Dict
from openai import OpenAI
from uuid import uuid4
from pydantic import BaseModel, ValidationError

class ReasoningPaths(BaseModel):
    title: str
    steps: List[str]
    conclusion: str

class QueryRecord(BaseModel):
    id: str
    prompt: str
    reasoning_style: Optional[str] = None
    paths: List[ReasoningPaths]

class AIReasoningSandbox:
    def __init__(self, model: str = "gpt-4o", temperature: float = 0.7):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.model = model
        self.temperature = temperature
        self.history: List[QueryRecord] = []

    def system_prompt(self, reasoning_style: Optional[str] = None) -> str:
        base_message = (
            "You are an expert in critical thinking, logical analysis, and structured reasoning. "
            "Your task is to analyze the user's prompt and generate multiple distinct reasoning paths that demonstrate diverse approaches to problem-solving."
        )
        if reasoning_style:
            base_message += f" Focus on applying a '{reasoning_style}' reasoning approach."
        return base_message

    def user_prompt(self, prompt: str, num_paths: int = 3) -> str:
        return (
            f"""
        You are an expert AI reasoning engine. Your task is to critically analyze the following scenario and generate {num_paths} distinct reasoning paths, each offering a unique and logically sound perspective.

        ### 📝 Task Overview:
        - Analyze the user's prompt using diverse reasoning approaches.
        - Structure each reasoning path with a clear title, step-by-step logical breakdown, and a final conclusion.

        ---

        ### 📌 User Prompt:
        #### {prompt.strip()}

        ---

        ### 🧠 Instructions for Each Reasoning Path:
        1. Title: Provide a short, descriptive title for the path (e.g., "Practical View", "Ethical Analysis").
        2. Steps: Present a numbered, logical sequence of reasoning steps. Each step should build on the previous one.
        3. Conclusion: Summarize the final insight, recommendation, or answer derived from the path.

        ---

        ### 💡 Output Format Example (JSON):

        [
          {{
            "title": "Practical View",
            "steps": [
              "Step one explanation...",
              "Step two explanation...",
              "Step three explanation..."
            ],
            "conclusion": "Final insight or answer."
          }},
          ...
        ]

        ---

        Now, generate {num_paths} well-structured reasoning paths for the prompt above and return ONLY a valid JSON array of objects in the format shown above. Do not include any explanations or markdown.
        """
        )

    def generate_response(self, prompt: str, num_paths: int = 3, reasoning_style: Optional[str] = None) -> List[ReasoningPaths]:
        system_prompt = self.system_prompt(reasoning_style)
        user_prompt = self.user_prompt(prompt, num_paths)
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=self.temperature
            )
            text = response.choices[0].message.content.strip()
            try:
                match = re.search(r'\[\s*{.*?}\s*\]', text, re.DOTALL)
                if match:
                    json_str = match.group(0)
                else:
                    json_str = text  
                paths_data = json.loads(json_str)
            except Exception as e:
                raise RuntimeError(f"Failed to parse JSON from model response: {e}\nRaw response:\n{text}")

            try:
                return [ReasoningPaths(**p) for p in paths_data]
            except ValidationError as ve:
                raise RuntimeError(f"Validation error: {ve}\nRaw data:\n{paths_data}")

        except Exception as e:
            raise RuntimeError(f"Error generating response: {str(e)}")

    def add_query(self, user_prompt: str, num_paths: int = 3, reasoning_style: Optional[str] = None) -> QueryRecord:
        paths = self.generate_response(user_prompt.strip(), num_paths, reasoning_style)
        record = QueryRecord(
            id=str(uuid4()),
            prompt=user_prompt,
            reasoning_style=reasoning_style,
            paths=paths
        )
        self.history.append(record)
        return record

    def get_history(self) -> List[Dict]:
        return [record.model_dump() for record in self.history]

    def clear_history(self) -> str:
        self.history = []
        return "History cleared."

## Step 4: Executing Reasoning Queries and Rendering Markdown Output
 
- The first line imports `Markdown` and `display` from `IPython.display` to render markdown output in a notebook environment.  

- An instance of `AIReasoningSandbox` is created and assigned to `sandbox`.  

- The `add_query` method is called on `sandbox` with a prompt about remote work, requesting 3 reasoning paths using the "Practical" style; the raw markdown response is stored in `response_md`.  

- Finally, `display(Markdown(response_md))` renders the AI-generated reasoning paths as formatted markdown output in the notebook.  


In [ ]:
from IPython.display import Markdown, display

sandbox = AIReasoningSandbox()
record = sandbox.add_query(
    "My female best friend blocked me on instagram and I don't know why. I want to understand her perspective and how to approach the situation.",
    num_paths=3,
    reasoning_style="Practical"
)

md = f"## **Prompt:**\n\n{record.prompt}\n\n"
for idx, path in enumerate(record.paths, 1):
    md += f"---\n\n"
    md += f"### 🧠 **Reasoning Path {idx}: {path.title}**\n\n"
    md += f"**Steps:**\n"
    for i, step in enumerate(path.steps, 1):
        md += f"{i}. {step}\n"
    md += f"\n**Conclusion:** {path.conclusion}\n\n"
md += "---"
display(Markdown(md))

## Conclusion :

The AI Reasoning Sandbox app offers a powerful and flexible platform for generating diverse reasoning pathways on any topic. Key highlights include:

- **Multi-style Reasoning:** Supports various reasoning styles (e.g., Practical, Ethical, Analytical) to approach problems from different angles.

- **Structured Outputs:** Produces clear, step-by-step reasoning paths with titles and conclusions, formatted in professional Markdown.

- **Customizable:** Allows users to specify the number of reasoning paths and preferred reasoning style for tailored analysis.

- **History Management:** Maintains a query history for easy tracking and review of past prompts and generated reasoning.

- **User-friendly Integration:** Simple interface for prompt input and markdown display, making results easy to read and share.

- **AI-powered Insight:** Leverages OpenAI’s models to enhance critical thinking and decision-making through diverse, logical perspectives.

Overall, the app is an effective tool for deepening understanding, fostering critical analysis, and supporting informed decisions with AI-augmented reasoning.


---

# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>